<!-- DCA Portfolio Relationship Notebook v4 -->
# VFX Production Intelligence Dashboard — Validate Table Relationships

Use the **Markdown task guide** for the complete workflow, schemas, key definitions,
result interpretation, and decision rules. Keep this notebook focused on SQL, outputs,
and your written conclusions.


In [84]:
%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

from pathlib import Path
import duckdb
import yaml

cwd = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents, cwd / "projects" / "project-01-vfx-production-intelligence")
        if (candidate / "config" / "project_sources.yaml").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Open this project's generated VS Code workspace, then run the setup cell again."
    )

config = yaml.safe_load(
    (project_root / "config" / "project_sources.yaml").read_text(encoding="utf-8")
) or {}
schema_name = str(config.get("schema") or "raw")

def qid(value):
    return '"' + str(value).replace('"', '""') + '"'

def qpath(value):
    return "'" + Path(value).resolve().as_posix().replace("'", "''") + "'"

def reader(path, source_format):
    source_path = qpath(path)
    if source_format == "csv":
        return f"read_csv_auto({source_path}, header = true)"
    if source_format == "parquet":
        return f"read_parquet({source_path})"
    if source_format == "json_lines":
        return f"read_json_auto({source_path}, format = 'newline_delimited')"
    if source_format == "json":
        return f"read_json_auto({source_path})"
    raise ValueError(f"Unsupported project source format: {source_format}")

con = duckdb.connect(":memory:")
con.execute(f"CREATE SCHEMA IF NOT EXISTS {qid(schema_name)}")
for source in config.get("sources") or []:
    source_path = project_root / source["path"]
    if not source_path.is_file():
        raise FileNotFoundError(f"Project source file is missing: {source_path}")
    con.execute(
        f"CREATE OR REPLACE VIEW {qid(schema_name)}.{qid(source['view'])} AS "
        f"SELECT * FROM {reader(source_path, source['format'])}"
    )

%sql con --alias project
print(f"Project data ready: {len(config.get('sources') or [])} tables loaded.")


The sql extension is already loaded. To reload it, use:
  %reload_ext sql
Project data ready: 6 tables loaded.


In [85]:
%%sql

SHOW ALL TABLES;

database,schema,name,column_names,column_types,temporary
memory,raw,artists,"['artist_id', 'artist_name', 'department', 'role', 'seniority', 'location', 'weekly_capacity_hours', 'hourly_cost_usd', 'hire_date', 'active_flag', 'manager_id', 'email']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'DATE', 'VARCHAR', 'VARCHAR', 'VARCHAR']",False
memory,raw,clients,"['client_id', 'client_name', 'client_type', 'region', 'contract_tier', 'revision_tendency_index', 'default_review_sla_hours', 'active_flag', 'account_manager', 'notes']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'DOUBLE', 'BIGINT', 'VARCHAR', 'VARCHAR', 'VARCHAR']",False
memory,raw,projects,"['project_id', 'client_id', 'project_name', 'project_type', 'start_date', 'target_delivery_date', 'actual_delivery_date', 'status', 'priority', 'planned_shot_count', 'budget_hours', 'producer', 'vfx_supervisor', 'fps', 'resolution', 'delivery_format']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'DATE', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'DOUBLE', 'VARCHAR', 'VARCHAR']",False
memory,raw,reviews,"['review_id', 'shot_id', 'project_id', 'review_date', 'review_round', 'review_type', 'feedback_source', 'outcome', 'note_category', 'response_hours', 'reviewer', 'notes']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR']",False
memory,raw,shots,"['shot_id', 'project_id', 'sequence_code', 'shot_name', 'department', 'assigned_artist_id', 'assigned_date', 'start_date', 'deadline', 'delivery_date', 'first_pass_date', 'final_approval_date', 'status', 'priority', 'complexity', 'estimated_hours', 'actual_hours', 'internal_revision_count', 'client_revision_count', 'revision_count_reported', 'hold_days', 'vendor_dependencies', 'notes']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'BIGINT', 'BIGINT', 'BIGINT', 'VARCHAR', 'VARCHAR']",False
memory,raw,time_entries,"['time_entry_id', 'shot_id', 'project_id', 'artist_id', 'work_date', 'hours_logged', 'task_type', 'billable_flag', 'overtime_flag', 'source_system', 'comment']","['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR']",False


The project setup loaded six source tables into the raw schema: artists, clients, projects, reviews, shots, and time entries. These raw tables will be evaluated without modifying their contents.

# 1. Baseline row counts and table grain


In [86]:
%%sql

SELECT 'shots' AS table_name, COUNT(*) AS row_count
FROM raw.shots

UNION ALL

SELECT 'clients' AS table_name, COUNT(*) AS row_count
FROM raw.clients

UNION ALL

SELECT 'projects' AS table_name, COUNT(*) AS row_count
FROM raw.projects

UNION ALL

SELECT 'reviews' AS table_name, COUNT(*) AS row_count
FROM raw.reviews

UNION ALL

SELECT 'artists' AS table_name, COUNT(*) AS row_count
FROM raw.artists

UNION ALL

SELECT 'time entries' AS table_name, COUNT(*) AS row_count
FROM raw.time_entries

ORDER BY table_name;


table_name,row_count
artists,62
clients,14
projects,16
reviews,3883
shots,1136
time entries,6935


### Interpretation

Baseline row counts were established for all six source tables. The data contains 62 artists, 14 clients, 16 projects, 3883 reviews, 1136 shots, and 6935 time entries. I plan to use these counts to identify unexpected row loss or multiplication during testing.


## Table Grain

| Table          | Expected grain                |
| -------------- | ----------------------------- |
| `artists`      | One row per artist            |
| `clients`      | One row per client            |
| `projects`     | One row per project           |
| `shots`        | One row per shot              |
| `reviews`      | One row per review event      |
| `time_entries` | One row per logged work entry |

The expected table grains are one row per artist, client, project, shot, review event, and time-entry event. Entity tables should have unique identifiers, while activity tables may legitimately contain repeated project, shot, or artist foreign keys.


# 2. Primary-key validity


## raw.artists

In [87]:
%%sql

SELECT COUNT(*) as total_rows,
COUNT(DISTINCT artist_id) AS distinct_ids,
SUM(
    CASE
        WHEN artist_id IS NULL THEN 1
        ELSE 0
    END
) AS null_ids
FROM raw.artists;


total_rows,distinct_ids,null_ids
62,60,0


In [88]:
%%sql

SELECT
    artist_id,
    COUNT(*) AS row_count
FROM raw.artists
GROUP BY artist_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, artist_id;

artist_id,row_count
ART-044,2
ART-053,2


## raw.clients

In [89]:
%%sql

SELECT COUNT(*) as total_rows,
COUNT(DISTINCT client_id) AS distinct_ids,
SUM(
    CASE
        WHEN client_id IS NULL THEN 1
        ELSE 0
    END
) AS null_ids
FROM raw.clients;

total_rows,distinct_ids,null_ids
14,12,0


In [90]:
%%sql

SELECT
    client_id,
    COUNT(*) AS row_count
FROM raw.clients
GROUP BY client_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, client_id;

client_id,row_count
CL-007,2
CL-012,2


## raw.projects

In [91]:
%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT project_id) AS distinct_ids,
    SUM(
        CASE
            WHEN project_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_ids
FROM raw.projects;

total_rows,distinct_ids,null_ids
16,15,0


In [92]:
%%sql

SELECT
    project_id,
    COUNT(*) AS row_count
FROM raw.projects
GROUP BY project_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, project_id;

project_id,row_count
PRJ-1005,2


## raw.shots

In [93]:
%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT shot_id) AS distinct_ids,
    SUM(
        CASE
            WHEN shot_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_ids
FROM raw.shots;

total_rows,distinct_ids,null_ids
1136,1112,0


In [94]:
%%sql

SELECT
    shot_id,
    COUNT(*) AS row_count
FROM raw.shots
GROUP BY shot_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, shot_id;

shot_id,row_count
PRJ-1001_SQ010_SH0040,2
PRJ-1001_SQ050_SH0070,2
PRJ-1001_SQ080_SH0060,2
PRJ-1004_SQ040_SH0070,2
PRJ-1004_SQ070_SH0030,2
PRJ-1004_SQ100_SH0020,2
PRJ-1006_SQ110_SH0040,2
PRJ-1007_SQ040_SH0060,2
PRJ-1008_SQ050_SH0010,2
PRJ-1009_SQ050_SH0060,2


### Interpretation

#### `raw.artists`
The table contains 62 rows and 60 distinct `artist_id` values, with no null IDs. Two rows exceed the distinct-ID count, and the duplicate query identifies `ART-044` and `ART-053` as repeated identifiers. Because the expected grain is one row per artist, the primary key does not currently support the expected grain.

#### `raw.clients`
The table contains 14 rows and 12 distinct `client_id` values, with no null IDs. Two rows exceed the distinct-ID count, and `CL-007` and `CL-012` are repeated identifiers. Because the expected grain is one row per client, the primary key does not currently support the expected grain.

#### `raw.projects`
The table contains 16 rows and 15 distinct `project_id` values, with no null IDs. One row exceeds the distinct-ID count, and `PRJ-1005` is repeated. Because the expected grain is one row per project, the primary key does not currently support the expected grain.

#### `raw.shots`
The table contains 1,136 rows and 1,112 distinct `shot_id` values, with no null IDs. There are 24 rows beyond the distinct-ID count. Because the expected grain is one row per shot, the repeated identifiers must be investigated during cleaning.

The raw entity tables contain no null primary keys, but all four contain duplicate identifiers. Their current primary keys are therefore not safe to use as unique parent keys in joins.

## raw.reviews

In [95]:
%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT review_id) AS distinct_ids,
    SUM(
        CASE
            WHEN review_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_ids
FROM raw.reviews;

total_rows,distinct_ids,null_ids
3883,3843,0


In [96]:
%%sql

SELECT
    review_id,
    COUNT(*) AS row_count
FROM raw.reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, review_id;

review_id,row_count
REV-0000204,2
REV-0000206,2
REV-0000254,2
REV-0000311,2
REV-0000349,2
REV-0000491,2
REV-0000516,2
REV-0000626,2
REV-0000695,2
REV-0000896,2


## raw.time_entries

In [97]:
%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT time_entry_id) AS distinct_ids,
    SUM(
        CASE
            WHEN time_entry_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_ids
FROM raw.time_entries;

total_rows,distinct_ids,null_ids
6935,6865,0


In [98]:
%%sql

SELECT
    time_entry_id,
    COUNT(*) AS row_count
FROM raw.time_entries
GROUP BY time_entry_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC, time_entry_id;

time_entry_id,row_count
TE-0000194,2
TE-0000208,2
TE-0000217,2
TE-0000311,2
TE-0000350,2
TE-0000527,2
TE-0000747,2
TE-0000759,2
TE-0000876,2
TE-0001098,2


### Activity-table primary-key interpretation

#### `raw.reviews`
The table contains 3,883 rows and 3,843 distinct `review_id` values, with no null IDs. There are 40 rows beyond the distinct-ID count. Because the expected grain is one row per review event, the repeated review IDs require investigation during cleaning.

#### `raw.time_entries`
The table contains 6,935 rows and 6,865 distinct `time_entry_id` values, with no null IDs. There are 70 rows beyond the distinct-ID count. Because the expected grain is one row per logged work event, the repeated time-entry IDs require investigation during cleaning.

# 3. Foreign-key validity


### Projects -> Clients

In [99]:
%%sql

SELECT
    COUNT(*) AS null_client_ids
FROM raw.projects
WHERE client_id IS NULL;


null_client_ids
0


In [100]:
%%sql

SELECT
    p.project_id,
    p.client_id
FROM raw.projects AS p
LEFT JOIN raw.clients AS c
    ON p.client_id = c.client_id
WHERE
    p.client_id IS NOT NULL
    AND c.client_id IS NULL;

project_id,client_id
PRJ-1011,CL-999
PRJ-1003,CL-004


### Shots -> Projects

In [101]:
%%sql

SELECT
    COUNT(*) AS null_project_ids
FROM raw.shots
WHERE project_id IS NULL;

null_project_ids
0


In [102]:
%%sql

SELECT COUNT(*) as orphan_count
FROM (
    SELECT
        s.shot_id,
        s.project_id
    FROM raw.shots AS s
    LEFT JOIN raw.projects AS p
        ON s.project_id = p.project_id
    WHERE
        s.project_id IS NOT NULL
        AND p.project_id IS NULL
)

orphan_count
18


### Reviews -> Shots

In [103]:
%%sql

SELECT
    COUNT(*) AS null_shot_ids
FROM raw.reviews
WHERE shot_id IS NULL;

null_shot_ids
9


In [104]:
%%sql

SELECT COUNT(*) AS orphan_count
FROM (
    SELECT
        r.*
    FROM raw.reviews AS r
    LEFT JOIN raw.shots AS s
        ON r.shot_id = s.shot_id
    WHERE
        r.shot_id IS NOT NULL
        AND s.shot_id IS NULL
)

orphan_count
26


### Time Entries -> Shots

In [105]:
%%sql

SELECT
    COUNT(*) AS null_shot_ids
FROM raw.time_entries
WHERE shot_id IS NULL;

null_shot_ids
12


In [106]:
%%sql

SELECT COUNT(*) AS orphan_count
FROM (
    SELECT
        t.*
    FROM raw.time_entries AS t
    LEFT JOIN raw.shots AS s
        ON t.shot_id = s.shot_id
    WHERE
        t.shot_id IS NOT NULL
        AND s.shot_id IS NULL
)

orphan_count
33


### Time Entries -> Artists

In [107]:
%%sql

SELECT
    COUNT(*) AS null_artist_ids
FROM raw.time_entries
WHERE artist_id IS NULL;

null_artist_ids
14


In [108]:
%%sql

SELECT COUNT(*) as orphan_count
FROM (
    SELECT
    t.*
FROM raw.time_entries AS t
LEFT JOIN raw.artists AS a
    ON t.artist_id = a.artist_id
WHERE
    t.artist_id IS NOT NULL
    AND a.artist_id IS NULL
)

orphan_count
19


### Interpretation

#### Projects → Clients
`projects.client_id` contains no null values, but two projects reference client IDs that do not exist in `raw.clients`. These projects cannot currently be connected to valid client records.

#### Shots → Projects
`shots.project_id` contains no null values, but 18 shots reference project IDs that do not exist in `raw.projects`. These shots cannot currently be connected to valid project records.

#### Reviews → Shots
Nine reviews have null `shot_id` values and 26 additional reviews reference shot IDs that do not exist in `raw.shots`. This produces 35 unmatched review records in a left join.

#### Time Entries → Shots
Twelve time entries have null `shot_id` values and 33 additional time entries reference shot IDs that do not exist in `raw.shots`. This produces 45 unmatched time-entry records in a left join.

#### Time Entries → Artists
Fourteen time entries have null `artist_id` values and 19 additional time entries reference artist IDs that do not exist in `raw.artists`. This produces 33 unmatched time-entry records in a left join.

Every tested foreign-key relationship contains at least one invalid or missing reference. These issues should be documented now and resolved or explicitly handled during cleaning.

# 4. Join cardinality


In [109]:
%%sql

SELECT
    (SELECT COUNT(*) FROM raw.projects) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE c.client_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.projects
    ) AS extra_rows
FROM raw.projects AS p
LEFT JOIN raw.clients AS c
    ON p.client_id = c.client_id;


child_rows,joined_rows,unmatched_rows,extra_rows
16,19,2,3


In [110]:
%%sql

SELECT
    (SELECT COUNT(*) FROM raw.shots) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE p.project_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.shots
    ) AS extra_rows
FROM raw.shots AS s
LEFT JOIN raw.projects AS p
    ON s.project_id = p.project_id;

child_rows,joined_rows,unmatched_rows,extra_rows
1136,1199,18,63


In [111]:
%%sql

SELECT
    (SELECT COUNT(*) FROM raw.reviews) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE s.shot_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.reviews
    ) AS extra_rows
FROM raw.reviews AS r
LEFT JOIN raw.shots AS s
    ON r.shot_id = s.shot_id;

child_rows,joined_rows,unmatched_rows,extra_rows
3883,3966,35,83


In [112]:
%%sql

SELECT
    (SELECT COUNT(*) FROM raw.time_entries) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE s.shot_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.time_entries
    ) AS extra_rows
FROM raw.time_entries AS t
LEFT JOIN raw.shots AS s
    ON t.shot_id = s.shot_id;

child_rows,joined_rows,unmatched_rows,extra_rows
6935,7100,45,165


In [113]:
%%sql

SELECT
    (SELECT COUNT(*) FROM raw.time_entries) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE a.artist_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.time_entries
    ) AS extra_rows
FROM raw.time_entries AS t
LEFT JOIN raw.artists AS a
    ON t.artist_id = a.artist_id;

child_rows,joined_rows,unmatched_rows,extra_rows
6935,7133,33,198


In [114]:
%%sql

SELECT
    'projects_to_clients' AS relationship,
    (SELECT COUNT(*) FROM raw.projects) AS child_rows,
    COUNT(*) AS joined_rows,
    COUNT(*) FILTER (
        WHERE c.client_id IS NULL
    ) AS unmatched_rows,
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.projects
    ) AS extra_rows
FROM raw.projects AS p
LEFT JOIN raw.clients AS c
    ON p.client_id = c.client_id

UNION ALL

SELECT
    'shots_to_projects',
    (SELECT COUNT(*) FROM raw.shots),
    COUNT(*),
    COUNT(*) FILTER (
        WHERE p.project_id IS NULL
    ),
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.shots
    )
FROM raw.shots AS s
LEFT JOIN raw.projects AS p
    ON s.project_id = p.project_id

UNION ALL

SELECT
    'reviews_to_shots',
    (SELECT COUNT(*) FROM raw.reviews),
    COUNT(*),
    COUNT(*) FILTER (
        WHERE s.shot_id IS NULL
    ),
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.reviews
    )
FROM raw.reviews AS r
LEFT JOIN raw.shots AS s
    ON r.shot_id = s.shot_id

UNION ALL

SELECT
    'time_entries_to_shots',
    (SELECT COUNT(*) FROM raw.time_entries),
    COUNT(*),
    COUNT(*) FILTER (
        WHERE s.shot_id IS NULL
    ),
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.time_entries
    )
FROM raw.time_entries AS t
LEFT JOIN raw.shots AS s
    ON t.shot_id = s.shot_id

UNION ALL

SELECT
    'time_entries_to_artists',
    (SELECT COUNT(*) FROM raw.time_entries),
    COUNT(*),
    COUNT(*) FILTER (
        WHERE a.artist_id IS NULL
    ),
    COUNT(*) - (
        SELECT COUNT(*) FROM raw.time_entries
    )
FROM raw.time_entries AS t
LEFT JOIN raw.artists AS a
    ON t.artist_id = a.artist_id

ORDER BY relationship;

relationship,child_rows,joined_rows,unmatched_rows,extra_rows
projects_to_clients,16,19,2,3
reviews_to_shots,3883,3966,35,83
shots_to_projects,1136,1199,18,63
time_entries_to_artists,6935,7133,33,198
time_entries_to_shots,6935,7100,45,165


### Interpretation

#### Projects → Clients
The join returned 19 rows from 16 project records, with two unmatched projects and three additional rows. Missing parent records and duplicate client keys make this relationship unsafe in its current raw form.

#### Shots → Projects
The join returned 1,199 rows from 1,136 shot records, with 18 unmatched shots and 63 additional rows. Missing parent records and duplicate project keys make this relationship unsafe in its current raw form.

#### Reviews → Shots
The join returned 3,966 rows from 3,883 review records, with 35 unmatched reviews and 83 additional rows. Missing shot references and duplicate shot keys make this relationship unsafe in its current raw form.

#### Time Entries → Shots
The join returned 7,100 rows from 6,935 time-entry records, with 45 unmatched entries and 165 additional rows. Missing shot references and duplicate shot keys make this relationship unsafe in its current raw form.

#### Time Entries → Artists
The join returned 7,133 rows from 6,935 time-entry records, with 33 unmatched entries and 198 additional rows. Missing artist references and duplicate artist keys make this relationship unsafe in its current raw form.

None of the five tested raw relationships is safe for final analytical joins. Every relationship either leaves child records unmatched, multiplies child records, or does both.

# 5. Project-specific consistency


In [115]:
%%sql

SELECT
    COUNT(*) AS total_reviews,
    COUNT(*) FILTER (
        WHERE r.shot_id IS NULL
           OR NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                WHERE s.shot_id = r.shot_id
           )
    ) AS unresolved_shot,
    COUNT(*) FILTER (
        WHERE EXISTS (
                SELECT 1
                FROM raw.shots AS s
                WHERE s.shot_id = r.shot_id
        )
          AND NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                WHERE s.shot_id = r.shot_id
          )
    ) AS unresolved_project,
    COUNT(*) FILTER (
        WHERE EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                WHERE s.shot_id = r.shot_id
        )
          AND NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                INNER JOIN raw.clients AS c
                    ON p.client_id = c.client_id
                WHERE s.shot_id = r.shot_id
          )
    ) AS unresolved_client
FROM raw.reviews AS r;

total_reviews,unresolved_shot,unresolved_project,unresolved_client
3883,35,60,377


In [116]:
%%sql

SELECT
    COUNT(*) AS total_time_entries,
    COUNT(*) FILTER (
        WHERE t.shot_id IS NULL
           OR NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                WHERE s.shot_id = t.shot_id
           )
    ) AS unresolved_shot,
    COUNT(*) FILTER (
        WHERE EXISTS (
                SELECT 1
                FROM raw.shots AS s
                WHERE s.shot_id = t.shot_id
        )
          AND NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                WHERE s.shot_id = t.shot_id
          )
    ) AS unresolved_project,
    COUNT(*) FILTER (
        WHERE EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                WHERE s.shot_id = t.shot_id
        )
          AND NOT EXISTS (
                SELECT 1
                FROM raw.shots AS s
                INNER JOIN raw.projects AS p
                    ON s.project_id = p.project_id
                INNER JOIN raw.clients AS c
                    ON p.client_id = c.client_id
                WHERE s.shot_id = t.shot_id
          )
    ) AS unresolved_client,
    COUNT(*) FILTER (
        WHERE t.artist_id IS NULL
           OR NOT EXISTS (
                SELECT 1
                FROM raw.artists AS a
                WHERE a.artist_id = t.artist_id
           )
    ) AS unresolved_artist
FROM raw.time_entries AS t;

total_time_entries,unresolved_shot,unresolved_project,unresolved_client,unresolved_artist
6935,45,109,718,33


### Time-entry Project Consistency

In [117]:
%%sql

SELECT
    table_name,
    column_name
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name IN ('reviews', 'time_entries')
  AND column_name = 'project_id'
ORDER BY table_name;

table_name,column_name
reviews,project_id
time_entries,project_id


In [118]:
%%sql

SELECT
    COUNT(*) AS inconsistent_time_entry_projects
FROM raw.time_entries AS t
WHERE EXISTS (
        SELECT 1
        FROM raw.shots AS s
        WHERE s.shot_id = t.shot_id
)
  AND NOT EXISTS (
        SELECT 1
        FROM raw.shots AS s
        WHERE s.shot_id = t.shot_id
          AND t.project_id IS NOT DISTINCT FROM s.project_id
);

inconsistent_time_entry_projects
147


In [119]:
%%sql

SELECT
    COUNT(*) AS inconsistent_review_projects
FROM raw.reviews AS r
WHERE EXISTS (
        SELECT 1
        FROM raw.shots AS s
        WHERE s.shot_id = r.shot_id
)
  AND NOT EXISTS (
        SELECT 1
        FROM raw.shots AS s
        WHERE s.shot_id = r.shot_id
          AND r.project_id IS NOT DISTINCT FROM s.project_id
);

inconsistent_review_projects
86


In [120]:
%%sql

SELECT
    table_name,
    column_name
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND column_name = 'client_id'
ORDER BY table_name;

table_name,column_name
clients,client_id
projects,client_id


In [121]:
%%sql

SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name IN ('shots', 'reviews')
  AND (
      column_name ILIKE '%_date'
      OR column_name ILIKE 'date_%'
  )
ORDER BY table_name, column_name;

table_name,column_name,data_type
reviews,review_date,VARCHAR
shots,assigned_date,VARCHAR
shots,delivery_date,VARCHAR
shots,final_approval_date,VARCHAR
shots,first_pass_date,VARCHAR
shots,start_date,VARCHAR


In [122]:
%%sql

SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name IN ('shots', 'time_entries')
  AND (
      column_name ILIKE '%_date'
      OR column_name ILIKE 'date_%'
  )
ORDER BY table_name, column_name;

table_name,column_name,data_type
shots,assigned_date,VARCHAR
shots,delivery_date,VARCHAR
shots,final_approval_date,VARCHAR
shots,first_pass_date,VARCHAR
shots,start_date,VARCHAR
time_entries,work_date,VARCHAR


In [123]:
%%sql

SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name IN ('projects', 'shots')
  AND (
      column_name ILIKE '%_date'
      OR column_name ILIKE 'date_%'
  )
ORDER BY table_name, column_name;

table_name,column_name,data_type
projects,actual_delivery_date,VARCHAR
projects,start_date,VARCHAR
projects,target_delivery_date,DATE
shots,assigned_date,VARCHAR
shots,delivery_date,VARCHAR
shots,final_approval_date,VARCHAR
shots,first_pass_date,VARCHAR
shots,start_date,VARCHAR


### Interpretation

The original production-chain queries multiplied source records because duplicate parent keys caused one review or time entry to appear more than once. The corrected queries above use `EXISTS` and `NOT EXISTS`, so every source review or time entry is counted once.

After running the corrected cells, record:

- The number of reviews with an unresolved shot, project, or client path.
- The number of time entries with an unresolved shot, project, client, or artist path.
- The exact number of time entries whose stored `project_id` does not agree with any matching shot record.
- The exact number of reviews whose stored `project_id` does not agree with any matching shot record.

Any project-ID differences may be caused by nulls, incorrect IDs, capitalization, or leading/trailing spaces. These are consistency findings for the cleaning stage; they should not be corrected in the validation notebook.

# Final relationship-validation conclusion

## Relationships safe to use

None of the five tested relationships is safe to use directly from the raw tables for final KPI calculations:

- Projects → Clients
- Shots → Projects
- Reviews → Shots
- Time Entries → Shots
- Time Entries → Artists

Every tested join contains unmatched child records, unexpected row multiplication, or both. Using these raw joins could omit records or inflate counts, hours, costs, and other production metrics.

## Issues found

Primary-key validation found duplicate identifiers in all six source tables:

- Artists: 62 rows and 60 distinct IDs
- Clients: 14 rows and 12 distinct IDs
- Projects: 16 rows and 15 distinct IDs
- Shots: 1,136 rows and 1,112 distinct IDs
- Reviews: 3,883 rows and 3,843 distinct IDs
- Time entries: 6,935 rows and 6,865 distinct IDs

Foreign-key validation found:

- Two projects with nonexistent clients
- Eighteen shots with nonexistent projects
- Nine reviews with null shot IDs and 26 reviews with nonexistent shot IDs
- Twelve time entries with null shot IDs and 33 with nonexistent shot IDs
- Fourteen time entries with null artist IDs and 19 with nonexistent artist IDs

Cardinality validation found unexpected row multiplication in all five tested joins, ranging from three additional rows in projects-to-clients to 198 additional rows in time-entries-to-artists.

Project-specific checks also indicate that some stored project identifiers do not agree with the project assigned through the shot relationship. Run the corrected count cells above to record the exact source-row counts without join inflation.

## Required cleaning or documentation

Before final analysis:

1. Determine whether repeated primary-key records are exact duplicates or conflicting records.
2. Define an authoritative record-selection rule for duplicated entity IDs.
3. Standardize identifier formatting, including capitalization and surrounding whitespace.
4. Resolve, map, exclude, or explicitly document null and orphaned foreign keys.
5. Define which table is authoritative when an activity record's `project_id` conflicts with its shot's `project_id`.
6. Create cleaned or validated tables/views rather than changing the raw source data.
7. Re-run this complete relationship-validation workflow against the cleaned layer.

## Final decision

Exploratory profiling may continue with clear warnings, but the raw relationships should not be used for final aggregations, dashboard KPIs, or portfolio conclusions. The project should proceed to the cleaning stage, after which relationship validation must be run again. Final analysis can continue only when the cleaned joins preserve child-row counts and all remaining unmatched records are intentionally documented.

# Appendix — Automated Relationship Validation

This appendix repeats the primary-key, foreign-key, join-cardinality, and project-consistency checks using configuration-driven Python.

The automated report does not modify the raw data. It provides a repeatable validation result that can be rerun after the cleaning stage.

In [124]:
PRIMARY_KEY_RULES = [
    {
        "table": "artists",
        "key": ["artist_id"],
        "grain": "One row per artist",
    },
    {
        "table": "clients",
        "key": ["client_id"],
        "grain": "One row per client",
    },
    {
        "table": "projects",
        "key": ["project_id"],
        "grain": "One row per project",
    },
    {
        "table": "shots",
        "key": ["shot_id"],
        "grain": "One row per shot",
    },
    {
        "table": "reviews",
        "key": ["review_id"],
        "grain": "One row per review event",
    },
    {
        "table": "time_entries",
        "key": ["time_entry_id"],
        "grain": "One row per logged work entry",
    },
]


RELATIONSHIP_RULES = [
    {
        "name": "Projects → Clients",
        "child_table": "projects",
        "foreign_key": ["client_id"],
        "parent_table": "clients",
        "parent_key": ["client_id"],
        "null_allowed": False,
    },
    {
        "name": "Shots → Projects",
        "child_table": "shots",
        "foreign_key": ["project_id"],
        "parent_table": "projects",
        "parent_key": ["project_id"],
        "null_allowed": False,
    },
    {
        "name": "Reviews → Shots",
        "child_table": "reviews",
        "foreign_key": ["shot_id"],
        "parent_table": "shots",
        "parent_key": ["shot_id"],
        "null_allowed": False,
    },
    {
        "name": "Time Entries → Shots",
        "child_table": "time_entries",
        "foreign_key": ["shot_id"],
        "parent_table": "shots",
        "parent_key": ["shot_id"],
        "null_allowed": False,
    },
    {
        "name": "Time Entries → Artists",
        "child_table": "time_entries",
        "foreign_key": ["artist_id"],
        "parent_table": "artists",
        "parent_key": ["artist_id"],
        "null_allowed": False,
    },
]

print(
    f"Configured {len(PRIMARY_KEY_RULES)} primary-key checks "
    f"and {len(RELATIONSHIP_RULES)} relationship checks."
)

Configured 6 primary-key checks and 5 relationship checks.


In [125]:
from IPython.display import Markdown, display


def quote_identifier(value):
    """Safely quote a DuckDB table, schema, or column name."""
    return '"' + str(value).replace('"', '""') + '"'


def table_reference(table_name):
    """Return a schema-qualified table reference."""
    return (
        f"{quote_identifier(schema_name)}."
        f"{quote_identifier(table_name)}"
    )


def display_markdown_table(rows, columns):
    """Display a list of dictionaries as a Markdown table."""
    if not rows:
        display(Markdown("_No results._"))
        return

    header = "| " + " | ".join(columns) + " |"
    alignment = "| " + " | ".join("---" for _ in columns) + " |"

    body = []

    for row in rows:
        values = []

        for column in columns:
            value = row.get(column, "")
            text = str(value).replace("|", r"\|").replace("\n", " ")
            values.append(text)

        body.append("| " + " | ".join(values) + " |")

    display(
        Markdown(
            "\n".join(
                [
                    header,
                    alignment,
                    *body,
                ]
            )
        )
    )


def validate_primary_key(rule):
    """Check one table's candidate primary key."""
    table_name = rule["table"]
    key_columns = rule["key"]
    table_sql = table_reference(table_name)

    key_sql = ", ".join(
        quote_identifier(column)
        for column in key_columns
    )

    null_condition = " OR ".join(
        f"{quote_identifier(column)} IS NULL"
        for column in key_columns
    )

    non_null_condition = " AND ".join(
        f"{quote_identifier(column)} IS NOT NULL"
        for column in key_columns
    )

    total_rows, null_key_rows = con.execute(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            SUM(
                CASE
                    WHEN {null_condition} THEN 1
                    ELSE 0
                END
            ) AS null_key_rows
        FROM {table_sql}
        """
    ).fetchone()

    duplicate_key_groups, duplicate_extra_rows = con.execute(
        f"""
        SELECT
            COUNT(*) AS duplicate_key_groups,
            COALESCE(SUM(row_count - 1), 0) AS duplicate_extra_rows
        FROM (
            SELECT
                {key_sql},
                COUNT(*) AS row_count
            FROM {table_sql}
            WHERE {non_null_condition}
            GROUP BY {key_sql}
            HAVING COUNT(*) > 1
        ) AS duplicate_keys
        """
    ).fetchone()

    passed = (
        null_key_rows == 0
        and duplicate_key_groups == 0
    )

    return {
        "table": table_name,
        "grain": rule["grain"],
        "key": ", ".join(key_columns),
        "total_rows": int(total_rows),
        "null_key_rows": int(null_key_rows),
        "duplicate_key_groups": int(duplicate_key_groups),
        "duplicate_extra_rows": int(duplicate_extra_rows),
        "status": "PASS" if passed else "ISSUE",
    }


def validate_relationship(rule):
    """Check nulls, orphans, and row multiplication for one relationship."""
    child_table = rule["child_table"]
    parent_table = rule["parent_table"]

    foreign_keys = rule["foreign_key"]
    parent_keys = rule["parent_key"]

    child_sql = table_reference(child_table)
    parent_sql = table_reference(parent_table)

    join_condition = " AND ".join(
        (
            f"parent.{quote_identifier(parent_key)} = "
            f"child.{quote_identifier(foreign_key)}"
        )
        for foreign_key, parent_key in zip(
            foreign_keys,
            parent_keys,
            strict=True,
        )
    )

    null_condition = " OR ".join(
        f"child.{quote_identifier(column)} IS NULL"
        for column in foreign_keys
    )

    non_null_condition = " AND ".join(
        f"child.{quote_identifier(column)} IS NOT NULL"
        for column in foreign_keys
    )

    child_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {child_sql}
        """
    ).fetchone()[0]

    null_foreign_key_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {child_sql} AS child
        WHERE {null_condition}
        """
    ).fetchone()[0]

    orphan_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {child_sql} AS child
        WHERE {non_null_condition}
          AND NOT EXISTS (
              SELECT 1
              FROM {parent_sql} AS parent
              WHERE {join_condition}
          )
        """
    ).fetchone()[0]

    joined_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {child_sql} AS child
        LEFT JOIN {parent_sql} AS parent
            ON {join_condition}
        """
    ).fetchone()[0]

    extra_rows = joined_rows - child_rows
    unmatched_rows = null_foreign_key_rows + orphan_rows

    nulls_pass = (
        rule["null_allowed"]
        or null_foreign_key_rows == 0
    )

    passed = (
        nulls_pass
        and orphan_rows == 0
        and extra_rows == 0
    )

    return {
        "relationship": rule["name"],
        "child_rows": int(child_rows),
        "joined_rows": int(joined_rows),
        "null_fk_rows": int(null_foreign_key_rows),
        "orphan_rows": int(orphan_rows),
        "unmatched_rows": int(unmatched_rows),
        "extra_rows": int(extra_rows),
        "safe_to_use": "YES" if passed else "NO",
        "status": "PASS" if passed else "ISSUE",
    }


print("Automated validation functions are ready.")

Automated validation functions are ready.


In [126]:
primary_key_results = [
    validate_primary_key(rule)
    for rule in PRIMARY_KEY_RULES
]

relationship_results = [
    validate_relationship(rule)
    for rule in RELATIONSHIP_RULES
]


display(Markdown("## Automated primary-key results"))

display_markdown_table(
    primary_key_results,
    [
        "table",
        "key",
        "total_rows",
        "null_key_rows",
        "duplicate_key_groups",
        "duplicate_extra_rows",
        "status",
    ],
)


display(Markdown("## Automated relationship results"))

display_markdown_table(
    relationship_results,
    [
        "relationship",
        "child_rows",
        "joined_rows",
        "null_fk_rows",
        "orphan_rows",
        "unmatched_rows",
        "extra_rows",
        "safe_to_use",
    ],
)

## Automated primary-key results

| table | key | total_rows | null_key_rows | duplicate_key_groups | duplicate_extra_rows | status |
| --- | --- | --- | --- | --- | --- | --- |
| artists | artist_id | 62 | 0 | 2 | 2 | ISSUE |
| clients | client_id | 14 | 0 | 2 | 2 | ISSUE |
| projects | project_id | 16 | 0 | 1 | 1 | ISSUE |
| shots | shot_id | 1136 | 0 | 24 | 24 | ISSUE |
| reviews | review_id | 3883 | 0 | 40 | 40 | ISSUE |
| time_entries | time_entry_id | 6935 | 0 | 70 | 70 | ISSUE |

## Automated relationship results

| relationship | child_rows | joined_rows | null_fk_rows | orphan_rows | unmatched_rows | extra_rows | safe_to_use |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Projects → Clients | 16 | 19 | 0 | 2 | 2 | 3 | NO |
| Shots → Projects | 1136 | 1199 | 0 | 18 | 18 | 63 | NO |
| Reviews → Shots | 3883 | 3966 | 9 | 26 | 35 | 83 | NO |
| Time Entries → Shots | 6935 | 7100 | 12 | 33 | 45 | 165 | NO |
| Time Entries → Artists | 6935 | 7133 | 14 | 19 | 33 | 198 | NO |

In [127]:
review_chain = con.execute(
    f"""
    SELECT
        COUNT(*) AS total_reviews,

        SUM(
            CASE
                WHEN r.shot_id IS NULL
                  OR NOT EXISTS (
                      SELECT 1
                      FROM {table_reference("shots")} AS s
                      WHERE s.shot_id = r.shot_id
                  )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_shot,

        SUM(
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    WHERE s.shot_id = r.shot_id
                )
                 AND NOT EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    WHERE s.shot_id = r.shot_id
                )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_project,

        SUM(
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    WHERE s.shot_id = r.shot_id
                )
                 AND NOT EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    INNER JOIN {table_reference("clients")} AS c
                        ON p.client_id = c.client_id
                    WHERE s.shot_id = r.shot_id
                )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_client

    FROM {table_reference("reviews")} AS r
    """
).fetchone()


time_entry_chain = con.execute(
    f"""
    SELECT
        COUNT(*) AS total_time_entries,

        SUM(
            CASE
                WHEN t.shot_id IS NULL
                  OR NOT EXISTS (
                      SELECT 1
                      FROM {table_reference("shots")} AS s
                      WHERE s.shot_id = t.shot_id
                  )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_shot,

        SUM(
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    WHERE s.shot_id = t.shot_id
                )
                 AND NOT EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    WHERE s.shot_id = t.shot_id
                )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_project,

        SUM(
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    WHERE s.shot_id = t.shot_id
                )
                 AND NOT EXISTS (
                    SELECT 1
                    FROM {table_reference("shots")} AS s
                    INNER JOIN {table_reference("projects")} AS p
                        ON s.project_id = p.project_id
                    INNER JOIN {table_reference("clients")} AS c
                        ON p.client_id = c.client_id
                    WHERE s.shot_id = t.shot_id
                )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_client,

        SUM(
            CASE
                WHEN t.artist_id IS NULL
                  OR NOT EXISTS (
                      SELECT 1
                      FROM {table_reference("artists")} AS a
                      WHERE a.artist_id = t.artist_id
                  )
                THEN 1
                ELSE 0
            END
        ) AS unresolved_artist

    FROM {table_reference("time_entries")} AS t
    """
).fetchone()


time_entry_project_mismatches = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {table_reference("time_entries")} AS t
    WHERE EXISTS (
        SELECT 1
        FROM {table_reference("shots")} AS s
        WHERE s.shot_id = t.shot_id
    )
      AND NOT EXISTS (
        SELECT 1
        FROM {table_reference("shots")} AS s
        WHERE s.shot_id = t.shot_id
          AND t.project_id IS NOT DISTINCT FROM s.project_id
    )
    """
).fetchone()[0]


review_project_mismatches = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {table_reference("reviews")} AS r
    WHERE EXISTS (
        SELECT 1
        FROM {table_reference("shots")} AS s
        WHERE s.shot_id = r.shot_id
    )
      AND NOT EXISTS (
        SELECT 1
        FROM {table_reference("shots")} AS s
        WHERE s.shot_id = r.shot_id
          AND r.project_id IS NOT DISTINCT FROM s.project_id
    )
    """
).fetchone()[0]


consistency_results = [
    {
        "check": "Reviews with unresolved shot",
        "issue_count": int(review_chain[1]),
        "status": "PASS" if review_chain[1] == 0 else "ISSUE",
    },
    {
        "check": "Reviews with unresolved project path",
        "issue_count": int(review_chain[2]),
        "status": "PASS" if review_chain[2] == 0 else "ISSUE",
    },
    {
        "check": "Reviews with unresolved client path",
        "issue_count": int(review_chain[3]),
        "status": "PASS" if review_chain[3] == 0 else "ISSUE",
    },
    {
        "check": "Time entries with unresolved shot",
        "issue_count": int(time_entry_chain[1]),
        "status": "PASS" if time_entry_chain[1] == 0 else "ISSUE",
    },
    {
        "check": "Time entries with unresolved project path",
        "issue_count": int(time_entry_chain[2]),
        "status": "PASS" if time_entry_chain[2] == 0 else "ISSUE",
    },
    {
        "check": "Time entries with unresolved client path",
        "issue_count": int(time_entry_chain[3]),
        "status": "PASS" if time_entry_chain[3] == 0 else "ISSUE",
    },
    {
        "check": "Time entries with unresolved artist",
        "issue_count": int(time_entry_chain[4]),
        "status": "PASS" if time_entry_chain[4] == 0 else "ISSUE",
    },
    {
        "check": "Time-entry project ID disagrees with shot",
        "issue_count": int(time_entry_project_mismatches),
        "status": (
            "PASS"
            if time_entry_project_mismatches == 0
            else "ISSUE"
        ),
    },
    {
        "check": "Review project ID disagrees with shot",
        "issue_count": int(review_project_mismatches),
        "status": (
            "PASS"
            if review_project_mismatches == 0
            else "ISSUE"
        ),
    },
]


display(Markdown("## Automated project-consistency results"))

display_markdown_table(
    consistency_results,
    [
        "check",
        "issue_count",
        "status",
    ],
)

## Automated project-consistency results

| check | issue_count | status |
| --- | --- | --- |
| Reviews with unresolved shot | 35 | ISSUE |
| Reviews with unresolved project path | 60 | ISSUE |
| Reviews with unresolved client path | 377 | ISSUE |
| Time entries with unresolved shot | 45 | ISSUE |
| Time entries with unresolved project path | 109 | ISSUE |
| Time entries with unresolved client path | 718 | ISSUE |
| Time entries with unresolved artist | 33 | ISSUE |
| Time-entry project ID disagrees with shot | 147 | ISSUE |
| Review project ID disagrees with shot | 86 | ISSUE |

In [128]:
primary_key_issue_count = sum(
    result["status"] == "ISSUE"
    for result in primary_key_results
)

unsafe_relationships = [
    result["relationship"]
    for result in relationship_results
    if result["safe_to_use"] == "NO"
]

consistency_issue_count = sum(
    result["status"] == "ISSUE"
    for result in consistency_results
)

overall_passed = (
    primary_key_issue_count == 0
    and not unsafe_relationships
    and consistency_issue_count == 0
)

overall_status = (
    "PASS — relationships are ready for validated downstream use"
    if overall_passed
    else "ISSUES FOUND — cleaning and revalidation are required"
)

safe_relationships = [
    result["relationship"]
    for result in relationship_results
    if result["safe_to_use"] == "YES"
]

safe_text = (
    ", ".join(safe_relationships)
    if safe_relationships
    else "None"
)

unsafe_text = (
    ", ".join(unsafe_relationships)
    if unsafe_relationships
    else "None"
)

display(
    Markdown(
        f"""
## Automated validation summary

**Overall status:** {overall_status}

- Primary-key checks with issues: **{primary_key_issue_count} of {len(primary_key_results)}**
- Relationships safe to use: **{safe_text}**
- Relationships requiring remediation: **{unsafe_text}**
- Project-consistency checks with issues: **{consistency_issue_count} of {len(consistency_results)}**

The automated result is an audit of the raw data. It does not clean or modify any source records.
"""
    )
)


## Automated validation summary

**Overall status:** ISSUES FOUND — cleaning and revalidation are required

- Primary-key checks with issues: **6 of 6**
- Relationships safe to use: **None**
- Relationships requiring remediation: **Projects → Clients, Shots → Projects, Reviews → Shots, Time Entries → Shots, Time Entries → Artists**
- Project-consistency checks with issues: **9 of 9**

The automated result is an audit of the raw data. It does not clean or modify any source records.


In [129]:
import csv
from datetime import datetime


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

report_directory = (
    project_root
    / "reports"
    / "relationship_validation"
    / timestamp
)

report_directory.mkdir(
    parents=True,
    exist_ok=True,
)


def write_csv_report(path, rows):
    if not rows:
        return

    fieldnames = list(rows[0].keys())

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(rows)


write_csv_report(
    report_directory / "primary_key_results.csv",
    primary_key_results,
)

write_csv_report(
    report_directory / "relationship_results.csv",
    relationship_results,
)

write_csv_report(
    report_directory / "consistency_results.csv",
    consistency_results,
)


summary_lines = [
    "# Automated Relationship Validation",
    "",
    f"Run timestamp: `{timestamp}`",
    "",
    f"**Overall status:** {overall_status}",
    "",
    (
        f"- Primary-key checks with issues: "
        f"{primary_key_issue_count} of {len(primary_key_results)}"
    ),
    f"- Relationships safe to use: {safe_text}",
    f"- Relationships requiring remediation: {unsafe_text}",
    (
        f"- Project-consistency checks with issues: "
        f"{consistency_issue_count} of {len(consistency_results)}"
    ),
    "",
    "The automated check did not modify the raw source data.",
]

summary_path = report_directory / "summary.md"

summary_path.write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("Automated validation report created:")
print(report_directory)

Automated validation report created:
C:\Users\Dan\Documents\MEGA\Dev\GitHub\career-accelerator\projects\project-01-vfx-production-intelligence\reports\relationship_validation\20260721_134407


## Automated validation interpretation

The automated validation reproduced the findings from the manual relationship checks. All six candidate primary keys contained duplicate identifiers, although none contained null identifiers. As a result, the raw entity and activity identifiers cannot currently be treated as guaranteed unique keys.

All five tested relationships failed the safe-to-use criteria. Each relationship contained unmatched child records, unexpected row multiplication, or both. The projects-to-clients join produced 3 additional rows, shots-to-projects produced 63, reviews-to-shots produced 83, time-entries-to-shots produced 165, and time-entries-to-artists produced 198. These additional rows indicate that duplicate parent identifiers could inflate downstream counts, hours, costs, and KPI calculations.

The project-specific consistency checks found that 472 of 3,883 reviews, or 12.16%, could not be traced through a complete shot-to-project-to-client chain. The same problem affected 872 of 6,935 time entries, or 12.57%. In addition, 33 time entries could not be connected to a valid artist. Stored project identifiers also disagreed with shot-based project assignments for 147 time entries and 86 reviews.

The raw relationships are therefore not safe for final reporting or KPI calculations. The project should continue to the cleaning stage, where duplicate identifiers, missing parent references, inconsistent project assignments, and identifier-formatting problems will be investigated. The automated validation should be rerun against the cleaned data before final analysis begins.
